In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import duckdb
import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
from statsbombpy import sb

In [ ]:
COMP_IDS = [11, 7, 2, 12]
SEASON_ID = 27

season_matches = sb.matches(competition_id=2, season_id=27)
season_matches["match_date"] = pd.to_datetime(season_matches["match_date"])
season_matches = season_matches.sort_values(by = "match_date")

In [ ]:
QUERY = "SELECT * FROM events WHERE team_id = '1'"
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
pm_df = con.execute(QUERY).df()
con.close()

In [ ]:
TEST_QUERY = """
-- primary table
CREATE TEMP TABLE pass_distance AS SELECT
    player_id,
    first(player) AS player,
    avg(pass_end_x - x) AS average_pass_x_displacement,
    avg(abs(pass_end_y - y)) AS average_pass_norm_y_displacement,
    count(*) AS pass_total
FROM events
WHERE
    (
        (type = 'Pass') AND
        (position <> 'Goalkeeper')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE carry_distance AS SELECT
    player_id,
    first(player) AS player,
    avg(end_x - x) AS average_carry_x_displacement,
    avg(abs(end_y - y)) AS average_carry_norm_y_displacement,
    count(*) AS carry_total
FROM events
WHERE
    (
        (type = 'Complete Carry')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE player_rel AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    CASE
        WHEN type = 'Complete Carry' THEN end_x
        ELSE x
        END AS x,
    CASE
        WHEN type = 'Complete Carry' THEN end_y
        ELSE y
        END AS y
FROM
    events
WHERE
    (
        (type = 'Pass') OR
        (type = 'Shot') OR
        ((type = 'Complete Carry') AND (carry_end_outcome = 'Other'))
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rel"
CREATE TEMP TABLE player_rel_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rel_x,
    avg(y) AS avg_rel_y,
    avg(40 - ABS(y - 40)) AS avg_normrel_y,
    count(*) AS rel_total
FROM player_rel
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE player_rec AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    x,
    y,
FROM
    events
WHERE 
    (
        (type = 'Ball Receipt*') OR
        (type = 'Ball Recovery') OR
        ((type = '50/50') AND ("50_50".outcome.name = 'Won')) OR
        ((type = 'Duel') AND (duel_outcome = 'Won')) OR
        ((type = 'Interception') AND (interception_outcome = 'Won')) OR
        ((type = 'Pass') AND (pass_type = 'Recovery')) OR
        ((type = 'Shot') AND (shot_type = 'Recovery'))
    )
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
    AND (matchweek >= $1 AND matchweek <= $2);

-- secondary table from "player_rec"
CREATE TEMP TABLE player_rec_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rec_x,
    avg(y) AS avg_rec_y,
    avg(40 - ABS(y - 40)) AS avg_normrec_y,
    count(*) AS rec_total
FROM player_rec
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE key_pass AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS key_pass_total
FROM
    events
WHERE
    (
        (type = 'Pass') AND
        (pass_shot_assist)
    )
    AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE shot_count AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS shot_total
FROM events
WHERE type = 'Shot' AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;

-- primary table
CREATE TEMP TABLE reception AS
    SELECT
        player_id,
        match_id,
        team_id,
        player,
        type,
        match_seconds,
        play_pattern,
        x,
        y,
    FROM
        events
    WHERE
        (
            (type = 'Ball Receipt*') OR
            (type = 'Ball Recovery') OR
            ((type = '50/50') AND ("50_50".outcome.name = 'Won')) OR
            ((type = 'Duel') AND (duel_outcome = 'Won')) OR
            ((type = 'Interception') AND (interception_outcome = 'Won'))
        )
        AND (matchweek >= $1 AND matchweek <= $2)
    ORDER BY
        match_id, team_id, match_seconds;

-- primary table
CREATE TEMP TABLE outcome AS
    SELECT
        player_id,
        match_id,
        team_id,
        type,
        match_seconds
    FROM
        events
    WHERE
        (
            (type = 'Pass') OR
            (type = 'Shot') OR
            (type = 'Complete Carry')
        )
        AND (matchweek >= $1 AND matchweek <= $2)
    ORDER BY
        match_id, team_id, match_seconds;

-- secondary table from "reception" and "outcome"
CREATE TEMP TABLE reception_outcomes AS
SELECT
    l.player_id as player_id,
    l.match_id as match_id,
    l.team_id as team_id,
    l.player as player,
    l.type as type,
    l.match_seconds as match_seconds,
    l.play_pattern as play_pattern,
    l.x as x,
    l.y as y,
    r.type as reception_outcome
FROM reception l LEFT JOIN outcome r ON
    l.player_id = r.player_id AND
    l.match_id = r.match_id AND
    l.team_id = r.team_id AND
    l.match_seconds = r.match_seconds;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_pass_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Pass'
GROUP BY player_id;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_carry_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Complete Carry'
GROUP BY player_id;

-- tertiary table from "reception_outcomes"
CREATE TEMP TABLE outcome_shot_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Shot'
GROUP BY player_id;

-- returned table
SELECT
    l.player_id AS player_id,
    l.player AS player,
    l.average_pass_x_displacement AS average_pass_x_displacement,
    l.average_pass_norm_y_displacement AS average_pass_norm_y_displacement,
    l.pass_total AS pass_total,
    r.average_carry_x_displacement AS average_carry_x_displacement,
    r.average_carry_norm_y_displacement AS average_carry_norm_y_displacement,
    r.carry_total AS carry_total,
    a.avg_rel_x AS average_release_x,
    a.avg_rel_y AS average_release_y,
    a.avg_normrel_y AS average_norm_release_y,
    a.rel_total AS rel_total,
    b.avg_rec_x AS average_reception_x,
    b.avg_rec_y AS average_reception_y,
    b.avg_normrec_y AS average_norm_reception_y,
    b.rec_total AS rec_total,
    (c.key_pass_total / (c.key_pass_total + d.shot_total)) AS key_pass_shot_ratio,
    c.key_pass_total AS key_pass_total,
    d.shot_total AS shot_total,
    (oc.outcome_count/(op.outcome_count + oc.outcome_count + osh.outcome_count)) AS carry_shotpass_ratio,
    op.outcome_count AS outcome_pass_total,
    oc.outcome_count AS outcome_carry_total,
    osh.outcome_count AS outcome_shot_total,
FROM pass_distance l
INNER JOIN carry_distance r ON
    l.player_id = r.player_id AND
    l.player = r.player
INNER JOIN player_rel_avg a ON
    l.player_id = a.player_id AND
    l.player = a.player
INNER JOIN player_rec_avg b ON
    l.player_id = b.player_id AND
    l.player = b.player
INNER JOIN key_pass c ON
    l.player_id = c.player_id AND
    l.player = c.player
INNER JOIN shot_count d ON
    l.player_id = d.player_id AND
    l.player = d.player
INNER JOIN outcome_pass_count op ON
    l.player_id = op.player_id AND
    l.player = op.player
INNER JOIN outcome_carry_count oc ON
    l.player_id = oc.player_id AND
    l.player = oc.player
INNER JOIN outcome_shot_count osh ON
    l.player_id = osh.player_id AND
    l.player = osh.player
WHERE
    (a.rel_total >= $1 AND
    b.rec_total >= $1);
"""

In [ ]:
statements = [s.strip() for s in TEST_QUERY.split(';') if s.strip()]

setup_statements = statements[:-1]
final_query = statements[-1]

PRIMARY_TABLES = [0,1,2,4,6,7,8,9]
SECONDARY_TABLES = [3,5,10]
TERTIARY_TABLES = [11,12,13]

DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
for x in PRIMARY_TABLES:
    con.execute(setup_statements[x], [0,24])
for y in SECONDARY_TABLES:
    con.execute(setup_statements[y])
for z in TERTIARY_TABLES:
    con.execute(setup_statements[z])
train_df = con.execute(final_query, [25]).df()
con.close()

con = duckdb.connect(DB_PATH, read_only = True)
for x in PRIMARY_TABLES:
    con.execute(setup_statements[x], [25,40])
for y in SECONDARY_TABLES:
    con.execute(setup_statements[y])
for z in TERTIARY_TABLES:
    con.execute(setup_statements[z])
test_df = con.execute(final_query, [20]).df()
con.close()

In [ ]:
feature_cols = [
    "average_pass_x_displacement",
    "average_pass_norm_y_displacement",
    "average_carry_x_displacement",
    "average_carry_norm_y_displacement",
    "average_release_x",
    "average_norm_release_y",
    "average_reception_x",
    "average_norm_reception_y",
    "key_pass_shot_ratio",
    "carry_shotpass_ratio"
]

target_col = ["shot_usage"]

In [ ]:

check_df = pd.merge(train_df, test_df, on = "player_id", how = "inner")

In [ ]:
LINEUP_QUERY = """
SELECT teamsheet, count(DISTINCT match_id) AS appearances, first(team_id) AS team_id FROM events GROUP BY teamsheet;
"""
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
teamsheets = con.execute(LINEUP_QUERY).df()
con.close()

In [ ]:
TEAMS_PLAYED_FOR = """
SELECT player_id, count(DISTINCT team_id) AS teams_played_for, first(team_id) AS team_id, last(team_id) AS second_team_id FROM events GROUP BY player_id;
"""
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
tpf_df = con.execute(TEAMS_PLAYED_FOR).df()
con.close()

In [ ]:
exclude_players = tpf_df[tpf_df["teams_played_for"] > 1][["player_id", "team_id"]]
exc_train_df = train_df[~(train_df["player_id"].isin(exclude_players["player_id"]))]
train_pid = pd.merge(exc_train_df["player_id"], tpf_df[["player_id", "team_id"]], how = "left", on = "player_id")
train_pid["player_id"] = train_pid["player_id"].astype(int)
train_pid = train_pid.sort_values(by = "team_id").reset_index(drop=True)

In [ ]:
player_ids = train_pid.copy()
merged = pd.merge(left = player_ids, right = player_ids, how = "inner", on = "team_id")

player_id_pairs = merged[~(merged["player_id_x"] == merged["player_id_y"])]
df_pairs = player_id_pairs.copy()

In [ ]:
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
con.register("player_pairs", player_id_pairs)

QUERY = """
CREATE TEMP TABLE shot_count AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS shot_total
FROM events
WHERE type = 'Shot' AND (matchweek >= $1 AND matchweek <= $2)
GROUP BY player_id;
"""

In [ ]:
SHOT_COUNT = """
SELECT
    count(*) AS shot_total
FROM events
WHERE type = 'Shot' AND (matchweek >= $1 AND matchweek <= $2) AND player_id = $3 AND list_contains(teamsheet, $4)
GROUP BY player_id;
"""

SHOT_INVOLVEMENT = """
SELECT
    count(*) AS shot_involvements
FROM events
WHERE type = 'Shot' AND (matchweek >= $1 AND matchweek <= $2) AND list_has_all(teamsheet, [$3, $4])
GROUP BY type;
"""

In [ ]:
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
shot_count = con.execute(SHOT_COUNT, [0, 24, 3385, 3394]).fetchall()[0][0]
shot_involvement = con.execute(SHOT_INVOLVEMENT, [0, 24, 3385, 3394]).fetchall()[0][0]
con.close()

print(shot_count)
print(shot_involvement)
print (shot_count/shot_involvement)

In [ ]:
def get_pairshotusage(target : int, partner : int, con, start : int = 0, end : int = 24):
    shot_count = con.execute(SHOT_COUNT, [int(start), int(end), int(target), int(partner)]).fetchall()
    shot_involvement = con.execute(SHOT_INVOLVEMENT,  [int(start), int(end), int(target), int(partner)]).fetchall()
    if shot_count and shot_involvement:
        return shot_count[0][0]/shot_involvement[0][0], shot_count[0][0], shot_involvement[0][0]
    else:
        return None, None, None

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)
con.register("pairs_df", df_pairs)

VECTORIZED_QUERY = """
WITH shot_events AS (
    -- 1. Pre-filter only Shot events within matchweeks 0 to 24
    SELECT 
        player_id AS shooter_id,
        teamsheet
    FROM events
    WHERE type = 'Shot'
      AND matchweek BETWEEN 0 AND 24
),
pair_metrics AS (
    -- 2. Join 9,000 pairs against shot events where both players share the pitch
    SELECT 
        p.player_id_x,
        p.player_id_y,
        -- Shots taken by target player (player_id_x) while co-present
        COUNT(CASE WHEN s.shooter_id = p.player_id_x THEN 1 END) AS shot_count,
        -- Total shots taken while both were co-present on pitch
        COUNT(*) AS shot_involvement
    FROM pairs_df p
    JOIN shot_events s
       ON list_contains(s.teamsheet, p.player_id_x)
      AND list_contains(s.teamsheet, p.player_id_y)
    GROUP BY p.player_id_x, p.player_id_y
)
-- 3. Preserve all 9,000 pairs with 0 fallback
SELECT 
    p.player_id_x,
    p.player_id_y,
    COALESCE(m.shot_count, 0) AS shot_count,
    COALESCE(m.shot_involvement, 0) AS shot_involvement,
    CASE 
        WHEN COALESCE(m.shot_involvement, 0) > 0 
        THEN COALESCE(m.shot_count, 0) * 1.0 / m.shot_involvement 
        ELSE 0.0 
    END AS shot_usage
FROM pairs_df p
LEFT JOIN pair_metrics m 
    ON p.player_id_x = m.player_id_x 
   AND p.player_id_y = m.player_id_y;
"""

df_results = con.execute(VECTORIZED_QUERY).df()
con.close()

# Merge results back into your main pairs DataFrame
df_pairs = df_pairs.merge(
    df_results, on=["player_id_x", "player_id_y"], how="left"
)

In [ ]:
cols = [
    "player_id",
    "average_pass_x_displacement",
    "average_pass_norm_y_displacement",
    "average_carry_x_displacement",
    "average_carry_norm_y_displacement",
    "average_release_x",
    "average_norm_release_y",
    "average_reception_x",
    "average_norm_reception_y",
    "key_pass_shot_ratio",
    "carry_shotpass_ratio"
]
features_train = train_df[cols].copy()
features_train["player_id"] = features_train["player_id"].astype(int)

In [ ]:
df_pairs = df_pairs.merge(features_train.add_suffix("_1"), left_on="player_id_x",
    right_on="player_id_1",
    how="left",
)

df_pairs = df_pairs.merge(
    features_train.add_suffix("_2"),
    left_on="player_id_y",
    right_on="player_id_2",
    how="left",
)

In [ ]:
print(df_pairs.columns.tolist())

In [ ]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Silence deprecation warnings globally for clean output
warnings.filterwarnings("ignore", category=UserWarning)

# =========================================================
# 1. Feature Specifications
# =========================================================
p1_features = [
    "average_pass_x_displacement_1",
    "average_pass_norm_y_displacement_1",
    "average_carry_x_displacement_1",
    "average_carry_norm_y_displacement_1",
    "average_release_x_1",
    "average_norm_release_y_1",
    "average_reception_x_1",
    "average_norm_release_y_1",
    "key_pass_shot_ratio_1",
    "carry_shotpass_ratio_1"
]

p2_features = [
    "average_pass_x_displacement_2",
    "average_pass_norm_y_displacement_2",
    "average_carry_x_displacement_2",
    "average_carry_norm_y_displacement_2",
    "average_release_x_2",
    "average_norm_release_y_2",
    "average_reception_x_2",
    "average_norm_release_y_2",
    "key_pass_shot_ratio_2",
    "carry_shotpass_ratio_2"
]

# Ensure no duplicate names if any exist
p1_features = list(dict.fromkeys(p1_features))
p2_features = list(dict.fromkeys(p2_features))
all_features = p1_features + p2_features
target_col = "shot_usage"

# =========================================================
# 2. Corrected Cross-Validation Trainer Function
# =========================================================
def train_lightgbm_cv(df, feature_list, target_col="shot_usage", n_splits=5):
    groups = df["player_id_1"]
    gkf = GroupKFold(n_splits=n_splits)
    
    oof_predictions = np.zeros(len(df))
    models = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df[target_col], groups)):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        X_train, y_train = train_df[feature_list], train_df[target_col]
        X_val, y_val = val_df[feature_list], val_df[target_col]
        
        sample_weights = train_df["shot_count"] if "shot_count" in train_df.columns else None
        val_weights = val_df["shot_count"] if "shot_count" in val_df.columns else None

        model = lgb.LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=-1
        )
        
        # Robust validation set format compatible across all LightGBM versions
        eval_set = [(X_val, y_val)]
        eval_weights = [val_weights] if val_weights is not None else None

        model.fit(
            X_train, y_train,
            sample_weight=sample_weights,
            eval_set=eval_set,
            eval_sample_weight=eval_weights,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        
        oof_predictions[val_idx] = model.predict(X_val)
        models.append(model)
        
    return models, oof_predictions

# =========================================================
# 3. Execution & Comparison
# =========================================================
# A. Baseline Model (Player 1 Features Only)
print("Training Baseline (Player 1 features only)...")
_, oof_p1_only = train_lightgbm_cv(df_pairs, p1_features, target_col=target_col)

# B. Joint Model (Player 1 + Player 2 Features)
print("Training Joint Model (Player 1 + Player 2 features)...")
models_joint, oof_joint = train_lightgbm_cv(df_pairs, all_features, target_col=target_col)

# Metrics
r2_p1 = r2_score(df_pairs[target_col], oof_p1_only)
rmse_p1 = np.sqrt(mean_squared_error(df_pairs[target_col], oof_p1_only))

r2_joint = r2_score(df_pairs[target_col], oof_joint)
rmse_joint = np.sqrt(mean_squared_error(df_pairs[target_col], oof_joint))

print("\n" + "=" * 55)
print("             MODEL COMPARISON SUMMARY                    ")
print("=" * 55)
print(f" Baseline (P1 Features Only) : R² = {r2_p1:.4f} | RMSE = {rmse_p1:.4f}")
print(f" Joint Model (P1 + P2)       : R² = {r2_joint:.4f} | RMSE = {rmse_joint:.4f}")
print(f" Added Value of Partner      : ΔR² = +{r2_joint - r2_p1:.4f}")
print("=" * 55 + "\n")

# =========================================================
# 4. SHAP Feature Importance Visualization
# =========================================================
print("Generating SHAP Feature Importance Plot...")
X = df_pairs[all_features]
shap_vals = np.zeros(X.shape)

for model in models_joint:
    explainer = shap.TreeExplainer(model)
    shap_vals += explainer.shap_values(X) / len(models_joint)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X, feature_names=all_features, show=True)

In [ ]:
import numpy as np
import pandas as pd

def add_relational_features(df):
    data = df.copy()
    
    # 1. Territorial Distance between Receptions
    data["reception_spatial_distance"] = np.sqrt(
        (data["average_reception_x_1"] - data["average_reception_x_2"])**2 +
        (data["average_norm_reception_y_1"] - data["average_norm_reception_y_2"])**2
    )
    
    # 2. Territorial Distance between Releases
    data["release_spatial_distance"] = np.sqrt(
        (data["average_release_x_1"] - data["average_release_x_2"])**2 +
        (data["average_norm_release_y_1"] - data["average_norm_release_y_2"])**2
    )
    
    # 3. Positional Depth Difference (Is P1 further up the pitch than P2?)
    data["depth_difference_p1_minus_p2"] = (
        data["average_reception_x_1"] - data["average_reception_x_2"]
    )
    
    # 4. Creator-Finisher Synergy (Does P2 supply key passes to shooter P1?)
    data["p2_feeder_p1_finisher_synergy"] = (
        data["key_pass_shot_ratio_2"] * data["carry_shotpass_ratio_1"]
    )
    
    return data

# Prepare dataframe with relational features
df_pairs_relational = add_relational_features(df_pairs)

relational_feature_cols = p1_features + [
    "reception_spatial_distance",
    "release_spatial_distance",
    "depth_difference_p1_minus_p2",
    "p2_feeder_p1_finisher_synergy"
]

# Retrain and evaluate
models_relational, oof_relational = train_lightgbm_cv(
    df_pairs_relational, 
    relational_feature_cols, 
    target_col=target_col
)

r2_rel = r2_score(df_pairs[target_col], oof_relational)
rmse_rel = np.sqrt(mean_squared_error(df_pairs[target_col], oof_relational))

print("\n" + "=" * 55)
print("          RELATIONAL FEATURE MODEL COMPARISON            ")
print("=" * 55)
print(f" Baseline (P1 Only)          : R² = {r2_p1:.4f} | RMSE = {rmse_p1:.4f}")
print(f" Raw Joint (P1 + P2 Raw)     : R² = {r2_joint:.4f} | RMSE = {rmse_joint:.4f}")
print(f" Relational Joint (P1 + Rel) : R² = {r2_rel:.4f} | RMSE = {rmse_rel:.4f}")
print(f" Added Value of Interactions : ΔR² = +{r2_rel - r2_p1:.4f}")
print("=" * 55 + "\n")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# =========================================================
# 1. PyTorch Architecture Definition
# =========================================================
class PlayerPairMLP(nn.Module):
    def __init__(self, p1_dim, p2_dim, hidden_dim=32, latent_dim=16, dropout_rate=0.15):
        super(PlayerPairMLP, self).__init__()
        
        # Shared Encoder Layer for individual player profiles
        self.p1_encoder = nn.Sequential(
            nn.Linear(p1_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, latent_dim),
            nn.BatchNorm1d(latent_dim),
            nn.ReLU()
        )
        
        self.p2_encoder = nn.Sequential(
            nn.Linear(p2_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, latent_dim),
            nn.BatchNorm1d(latent_dim),
            nn.ReLU()
        )
        
        # Combined Interaction Head
        # Inputs: [e1, e2, (e1 - e2), (e1 * e2)] -> Total dimension = latent_dim * 4
        interaction_dim = latent_dim * 4
        
        self.head = nn.Sequential(
            nn.Linear(interaction_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()  # Restricts outputs smoothly to range [0, 1]
        )

    def forward(self, x1, x2):
        e1 = self.p1_encoder(x1)
        e2 = self.p2_encoder(x2)
        
        # Compute explicit difference & multiplicative synergy in latent space
        diff = e1 - e2
        prod = e1 * e2
        
        combined = torch.cat([e1, e2, diff, prod], dim=1)
        output = self.head(combined)
        return output.squeeze(-1)

# =========================================================
# 2. Cross-Validation Trainer Function
# =========================================================
def train_mlp_cv(df, p1_cols, p2_cols, target_col="shot_usage", n_splits=5, epochs=60, batch_size=64, lr=0.002):
    groups = df["player_id_1"]
    gkf = GroupKFold(n_splits=n_splits)
    
    oof_predictions = np.zeros(len(df))
    
    # Standardize input features across the entire dataset structure
    scaler_p1 = StandardScaler()
    scaler_p2 = StandardScaler()
    
    X1_scaled = scaler_p1.fit_transform(df[p1_cols])
    X2_scaled = scaler_p2.fit_transform(df[p2_cols])
    y_vals = df[target_col].values
    
    print(f"Training PyTorch Pairwise MLP across {n_splits} folds...")
    print("=" * 55)
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df[target_col], groups)):
        # Prepare Tensors
        X1_train, X2_train = torch.tensor(X1_scaled[train_idx], dtype=torch.float32), torch.tensor(X2_scaled[train_idx], dtype=torch.float32)
        y_train = torch.tensor(y_vals[train_idx], dtype=torch.float32)
        
        X1_val, X2_val = torch.tensor(X1_scaled[val_idx], dtype=torch.float32), torch.tensor(X2_scaled[val_idx], dtype=torch.float32)
        y_val = torch.tensor(y_vals[val_idx], dtype=torch.float32)
        
        # Dataloaders
        train_dataset = TensorDataset(X1_train, X2_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        # Model initialization
        model = PlayerPairMLP(p1_dim=len(p1_cols), p2_dim=len(p2_cols))
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.MSELoss()
        
        # Early Stopping setup
        best_val_loss = float("inf")
        best_preds = None
        
        # Training loop
        model.train()
        for epoch in range(epochs):
            for b_x1, b_x2, b_y in train_loader:
                optimizer.zero_grad()
                preds = model(b_x1, b_x2)
                loss = criterion(preds, b_y)
                loss.backward()
                optimizer.step()
            
            # Validation step
            model.eval()
            with torch.no_grad():
                val_preds = model(X1_val, X2_val)
                val_loss = criterion(val_preds, y_val).item()
                
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_preds = val_preds.numpy()
            model.train()
            
        oof_predictions[val_idx] = best_preds
        
        fold_r2 = r2_score(y_vals[val_idx], best_preds)
        fold_rmse = np.sqrt(mean_squared_error(y_vals[val_idx], best_preds))
        print(f"Fold {fold + 1} | Val RMSE: {fold_rmse:.4f} | Val R²: {fold_r2:.4f}")
        
    overall_r2 = r2_score(y_vals, oof_predictions)
    overall_rmse = np.sqrt(mean_squared_error(y_vals, oof_predictions))
    
    print("=" * 55)
    print(f" PyTorch Pairwise MLP Overall OOF Performance ")
    print(f" Overall R²   : {overall_r2:.4f}")
    print(f" Overall RMSE : {overall_rmse:.4f}")
    print("=" * 55 + "\n")
    
    return oof_predictions

# =========================================================
# 3. Execute Training
# =========================================================
oof_mlp = train_mlp_cv(
    df=df_pairs, 
    p1_cols=p1_features, 
    p2_cols=p2_features, 
    target_col=target_col,
    epochs=50,
    batch_size=64,
    lr=0.002
)

In [ ]:
"""from tqdm import tqdm

tqdm.pandas(desc="Processing pairs")

DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
player_id_pairs[["shot_usage", "shot_count", "shot_involvement"]] = player_id_pairs.progress_apply(lambda row: get_pairshotusage(row["player_id_x"], row["player_id_y"], con), axis=1)
con.close()
"""
